In [1]:
pip install metar


Note: you may need to restart the kernel to use updated packages.


In [3]:
def get_historique_metar(icao_code, heures=24):
    """Récupère l'historique METAR sur plusieurs heures."""
    url = f"https://aviationweather.gov/api/data/metar?ids={icao_code}&format=json&hours={heures}"
    headers = {'User-Agent': 'AeroTwin-Project/1.0'}
    
    reponse = requests.get(url, headers=headers)
    reponse.raise_for_status()
    donnees = reponse.json()
    
    lignes_meteo = []
    for obs_json in donnees:
        metar_brut = obs_json.get('rawOb')
        try:
            obs = Metar.Metar(metar_brut)
            lignes_meteo.append({
                'aeroport': icao_code,
                'date_heure': obs.time,
                'temperature_c': obs.temp.value() if obs.temp else None,
                'vent_vitesse_kt': obs.wind_speed.value() if obs.wind_speed else None,
                'pression_hpa': obs.press.value() if obs.press else None,
                'metar_brut': metar_brut
            })
        except Exception:
            continue # Ignore les messages mal formatés
            
    return pd.DataFrame(lignes_meteo)

# Exécution pour Toulouse (24 dernières heures)
df_meteo_historique = get_historique_metar("LFBO", heures=24)
df_meteo_historique

,aeroport,date_heure,temperature_c,vent_vitesse_kt,pression_hpa,metar_brut
0,LFBO,2026-03-16 10:30:00,13.0,4.0,1025.0,METAR LFBO 161030Z AUTO 29004KT 210V340 9999 F...
1,LFBO,2026-03-16 10:00:00,13.0,3.0,1025.0,METAR LFBO 161000Z AUTO 24003KT 160V310 CAVOK ...
2,LFBO,2026-03-16 09:30:00,12.0,3.0,1025.0,METAR LFBO 160930Z AUTO 26003KT 200V320 CAVOK ...
3,LFBO,2026-03-16 09:00:00,11.0,3.0,1025.0,METAR LFBO 160900Z AUTO 27003KT 190V310 CAVOK ...
4,LFBO,2026-03-16 08:30:00,10.0,3.0,1025.0,METAR LFBO 160830Z AUTO VRB03KT CAVOK 10/05 Q1...
5,LFBO,2026-03-16 08:00:00,8.0,2.0,1025.0,METAR LFBO 160800Z AUTO VRB02KT CAVOK 08/04 Q1...
6,LFBO,2026-03-16 07:30:00,6.0,3.0,1024.0,METAR LFBO 160730Z AUTO VRB03KT CAVOK 06/04 Q1...
7,LFBO,2026-03-16 07:00:00,5.0,5.0,1025.0,METAR LFBO 160700Z AUTO 16005KT 9999 FEW037 05...
8,LFBO,2026-03-16 06:30:00,4.0,3.0,1025.0,METAR LFBO 160630Z AUTO 21003KT CAVOK 04/02 Q1...
9,LFBO,2026-03-16 06:00:00,5.0,6.0,1024.0,METAR LFBO 160600Z AUTO 24006KT 220V280 CAVOK ...


In [4]:
# Sauvegarde des données météo dans le dossier data
chemin_meteo = '../data/meteo_historique.csv'
df_meteo_historique.to_csv(chemin_meteo, index=False)
print(f"Météo sauvegardée dans {chemin_meteo}")

Météo sauvegardée dans ../data/meteo_historique.csv
